# 1. 기본 전처리

In [ ]:
!pip install -U ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.3 MB/s eta 0:00:00


In [ ]:
#데이터 불러오기
import pandas as pd
from ydata_profiling import ProfileReport
data = pd.read_csv('/content/dataset.csv')

/tmp/ipykernel_2475/2785852706.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [ ]:
# 타깃 피처 만들기
df = data[data['isOpen'] == 0].copy() # isOpen == 1인 경우 아직 거래가 완전히 끝나지 않은 불완전한 데이터이기 때문에 clear_date 값이 존재하지 않음
df['clear_date'] = pd.to_datetime(df['clear_date'], errors = 'coerce')
df['due_in_date'] = pd.to_datetime(df['due_in_date'].astype(int).astype(str),format='%Y%m%d' ,errors = 'coerce')
df['target'] = (df['clear_date'] > df['due_in_date']).astype(int)

In [ ]:
# 중복 행 제거
before_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed duplicate rows: {before_rows - len(df):,}")
print(f"Remaining rows: {len(df):,}")

Removed duplicate rows: 842
Remaining rows: 39,158


In [ ]:
# 학습 데이터, 테스트 데이터 나누기
df = df.sort_values('baseline_create_date').reset_index(drop=True) # 과거의 학습 데이터로 미래의 데이터를 분류해야하기 때문에 시간 순서로 정렬
split_idx = int(len(df) * 0.8) # train : test = 8 : 2로 분리
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

In [ ]:
# 학습 데이터, 테스트 데이터 다운로드
train_df.to_csv("/content/train.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)

In [ ]:
# 기초 EDA 정보 다운로드
report = ProfileReport(train_df)
report.to_file("/content/data_report.html")

/usr/local/lib/python3.12/dist-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 20/20 [00:03<00:00,  6.43it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# 2. 데이터 클랜징


In [ ]:
import pandas as pd
import numpy as np
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

In [ ]:
# 캐나다 달러와 US 달러 통일
train['amount_in_usd'] = train.apply(lambda row: row['total_open_amount'] * 0.75 if row['invoice_currency'] == 'CAD' else row['total_open_amount'], axis=1)
test['amount_in_usd'] = test.apply(lambda row: row['total_open_amount'] * 0.75 if row['invoice_currency'] == 'CAD' else row['total_open_amount'], axis=1)

In [ ]:
#total_open_amount의 분포가 매우 치우쳐져 있기 때문에, 로그 변환을 통해 분포 완화
train['amount_in_usd'] = np.log1p(train['amount_in_usd'])
test['amount_in_usd'] = np.log1p(test['amount_in_usd'])

In [ ]:
# 결제 수단의 등장 횟수가 30회 이하인 경우 Other 클래스로 변환
counts = train["cust_payment_terms"].value_counts()
train["cust_payment_terms_grp"] = train["cust_payment_terms"].where(
    train["cust_payment_terms"].map(counts) > 30,
    "Other"
)
test["cust_payment_terms_grp"] = test["cust_payment_terms"].where(
    test["cust_payment_terms"].map(counts) > 30,
    "Other"
)

In [ ]:
# 송장이 생성된 기준 날짜의 월, 일, 요일 추출
train['baseline_create_date'] = pd.to_datetime(train['baseline_create_date'], format='%Y%m%d', errors='coerce')
test['baseline_create_date'] = pd.to_datetime(test['baseline_create_date'], format='%Y%m%d', errors='coerce')

train['baseline_month'] = train['baseline_create_date'].dt.month
train['baseline_day'] = train['baseline_create_date'].dt.day
train['baseline_dayofweek'] = train['baseline_create_date'].dt.dayofweek
test['baseline_month'] = test['baseline_create_date'].dt.month
test['baseline_day'] = test['baseline_create_date'].dt.day
test['baseline_dayofweek'] = test['baseline_create_date'].dt.dayofweek


In [ ]:
# 송장 생성일과 마감일 사이의 기간 추가
train['due_in_date'] = pd.to_datetime(train['due_in_date'], errors='coerce')
test['due_in_date'] = pd.to_datetime(test['due_in_date'], errors='coerce')
train['Allowed_Pay_Days'] = (train['due_in_date'] - train['baseline_create_date']).dt.days
test['Allowed_Pay_Days'] = (test['due_in_date'] - test['baseline_create_date']).dt.days

In [ ]:
# cust_number 자릿수 맞추기
def clean_cust_number(df):
    df = df.copy()

    cust_number_stripped = df["cust_number"].astype(str).str.strip()
    is_9_digit = cust_number_stripped.str.fullmatch(r"\d{9}")

    df["cust_number"] = cust_number_stripped.where(
        ~is_9_digit,
        cust_number_stripped.str.zfill(10)
    )

    return df

train = clean_cust_number(train)
test = clean_cust_number(test)

In [ ]:
drop_cols = ['doc_id', 'invoice_currency', 'document type', 'area_business', 'isOpen', 'invoice_id','document_create_date', 'document_create_date.1', 'posting_date', 'total_open_amount']
train.drop(columns = drop_cols, inplace=True)
test.drop(columns = drop_cols, inplace=True)


In [ ]:
train.to_csv('/content/train_cleaned.csv', index=False)
test.to_csv('/content/test_cleaned.csv', index=False)

In [ ]:
train

,business_code,cust_number,name_customer,clear_date,buisness_year,due_in_date,posting_id,baseline_create_date,cust_payment_terms,target,amount_in_usd,cust_payment_terms_grp,baseline_month,baseline_day,baseline_dayofweek,Allowed_Pay_Days
0,CA02,0140104249,SOB associates,2019-01-23,2019.0,2018-12-24,1.0,2018-12-14,CA10,1,11.663576,CA10,12,14,4,10
1,U001,0200706844,WINC co,2019-02-19,2019.0,2018-12-29,1.0,2018-12-14,NAA8,1,6.359366,NAA8,12,14,4,15
2,U001,0200726979,BJ'S corp,2019-01-15,2019.0,2019-01-14,1.0,2018-12-30,NAA8,1,4.080584,NAA8,12,30,6,15
3,U001,0200706844,WINC systems,2019-01-15,2019.0,2019-01-14,1.0,2018-12-30,NAA8,1,10.962005,NAA8,12,30,6,15
4,U001,0200794332,COST foundation,2019-01-23,2019.0,2019-01-14,1.0,2018-12-30,NAAX,1,8.992506,NAAX,12,30,6,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31321,U001,0200788848,DAIRY trust,2019-11-27,2019.0,2019-12-03,1.0,2019-11-18,NAA8,0,10.639189,NAA8,11,18,0,15
31322,CA02,0140104249,SOB systems,2019-12-02,2019.0,2019-11-28,1.0,2019-11-18,CA10,1,10.771515,CA10,11,18,0,10
31323,U001,0200779051,AFFILI systems,2019-11-27,2019.0,2019-12-03,1.0,2019-11-18,NAA8,0,11.896931,NAA8,11,18,0,15
31324,U001,0200772670,ASSOCIAT co,2019-12-04,2019.0,2019-12-03,1.0,2019-11-18,NAU5,1,8.570087,NAU5,11,18,0,15


# 3. 통합 EDA

## 피처 엔지니어링 EDA

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT
TRAIN_PATH = DATA_DIR / "train_cleaned.csv"
TEST_PATH = DATA_DIR / "test_cleaned.csv"
TRAIN_OUT = DATA_DIR / "train_eda_revised.csv"
TEST_OUT = DATA_DIR / "test_eda_revised.csv"
MACRO_QUARTER_PATH = DATA_DIR / "macro_variable.csv"
MACRO_MARKET_PATH = DATA_DIR / "market_macro_monthly.csv"

DATE_COLS = ["baseline_create_date", "due_in_date", "clear_date"]
TEXT_DTYPES = {
    "cust_number": "string",
    "business_code": "string",
    "name_customer": "string",
    "cust_payment_terms": "string",
    "cust_payment_terms_grp": "string",
}

LATE_BUSINESS_DAYS = 5
# [기존 코드 - 주석처리] N_RECENT = 5
RECENT_HISTORY_WINDOWS = [5, 10, 20]
TIME_DECAY_LAMBDA = 0.8

## 데이터 로드

In [ ]:
def load_invoice_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=TEXT_DTYPES)
    if "target" in df.columns:
        df = df.rename(columns={"target": "target_old"})

    # 날짜 컬럼은 이후 영업일 계산에 바로 쓸 수 있게 변환한다.
    for col in DATE_COLS:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


train_eda_revised = load_invoice_data(TRAIN_PATH)
test_eda_revised = load_invoice_data(TEST_PATH)

for name, df in [("train", train_eda_revised), ("test", test_eda_revised)]:
    print(f"{name}: rows={len(df):,}, cols={len(df.columns)}, customers={df['cust_number'].nunique():,}")
    display(df.head(3))

train: rows=31,326, cols=16, customers=982


,business_code,cust_number,name_customer,clear_date,buisness_year,due_in_date,posting_id,baseline_create_date,cust_payment_terms,target_old,amount_in_usd,cust_payment_terms_grp,baseline_month,baseline_day,baseline_dayofweek,Allowed_Pay_Days
0,CA02,0140104249,SOB associates,2019-01-23,2019.0,2018-12-24,1.0,2018-12-14,CA10,1,11.663576,CA10,12,14,4,10
1,U001,0200706844,WINC co,2019-02-19,2019.0,2018-12-29,1.0,2018-12-14,NAA8,1,6.359366,NAA8,12,14,4,15
2,U001,0200726979,BJ'S corp,2019-01-15,2019.0,2019-01-14,1.0,2018-12-30,NAA8,1,4.080584,NAA8,12,30,6,15


test: rows=7,832, cols=16, customers=582


,business_code,cust_number,name_customer,clear_date,buisness_year,due_in_date,posting_id,baseline_create_date,cust_payment_terms,target_old,amount_in_usd,cust_payment_terms_grp,baseline_month,baseline_day,baseline_dayofweek,Allowed_Pay_Days
0,CA02,0140106408,WAL-M foundation,2019-12-05,2019.0,2019-11-28,1.0,2019-11-18,CA10,1,10.817586,CA10,11,18,0,10
1,U001,0200718130,SYSCO F trust,2019-12-23,2019.0,2019-12-20,1.0,2019-11-18,NA32,1,10.121198,NA32,11,18,0,32
2,U001,0200755701,ASSOCI foundation,2019-12-04,2019.0,2019-12-03,1.0,2019-11-18,NAA8,1,11.586048,NAA8,11,18,0,15


## 기본 피쳐 생성

In [ ]:
def business_days_between(start_dates, end_dates) -> pd.Series:
    start_dates = pd.to_datetime(start_dates, errors="coerce")
    if not isinstance(end_dates, pd.Series):
        end_dates = pd.Series(end_dates, index=start_dates.index)
    end_dates = pd.to_datetime(end_dates, errors="coerce")

    result = pd.Series(np.nan, index=start_dates.index, dtype="float")
    valid = start_dates.notna() & end_dates.notna()
    if valid.any():
        start_np = start_dates.loc[valid].values.astype("datetime64[D]")
        end_np = end_dates.loc[valid].values.astype("datetime64[D]")
        result.loc[valid] = np.busday_count(start_np, end_np)
    return result


def add_business_days_late(df: pd.DataFrame) -> None:
    # 주말을 제외한 실제 지연일을 계산한다.
    df["business_days_late"] = business_days_between(df["due_in_date"], df["clear_date"]).astype(int)


def add_basic_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> np.ndarray:
    for df in [train_df, test_df]:
        add_business_days_late(df)
        df["target"] = (df["business_days_late"] > LATE_BUSINESS_DAYS).astype(int)
        df["due_weekend_flag"] = df["due_in_date"].dt.weekday.isin([5, 6]).astype(int)

    # 금액 구간은 train 기준으로 만들고 test에는 같은 경계를 적용한다.
    _, raw_bins = pd.qcut(train_df["amount_in_usd"], q=4, labels=False, retbins=True, duplicates="drop")
    amount_bins = np.r_[-np.inf, raw_bins[1:-1], np.inf]
    amount_labels = list(range(len(amount_bins) - 1))

    for df in [train_df, test_df]:
        df["amount_bin"] = pd.cut(
            df["amount_in_usd"],
            bins=amount_bins,
            labels=amount_labels,
            include_lowest=True,
        ).astype("int64")

    return amount_bins


amount_bins = add_basic_features(train_eda_revised, test_eda_revised)
print("Amount bin edges:", amount_bins)
print(pd.DataFrame({
    "train_target_old": train_eda_revised["target_old"].value_counts().sort_index(),
    "train_target": train_eda_revised["target"].value_counts().sort_index(),
    "test_target_old": test_eda_revised["target_old"].value_counts().sort_index(),
    "test_target": test_eda_revised["target"].value_counts().sort_index(),
}).fillna(0).astype(int))

Amount bin edges: [       -inf  8.40600126  9.7273871  10.72211308         inf]
   train_target_old  train_target  test_target_old  test_target
0             17998         29242             4742         7409
1             13328          2084             3090          423


## 고객 이력 기반 피처

In [ ]:
def add_business_days(start_dates, n_business_days: int) -> pd.Series:
    start_dates = pd.to_datetime(start_dates, errors="coerce")
    result = pd.Series(pd.NaT, index=start_dates.index)
    valid = start_dates.notna()
    if valid.any():
        start_np = start_dates.loc[valid].values.astype("datetime64[D]")
        result.loc[valid] = pd.to_datetime(np.busday_offset(start_np, n_business_days, roll="forward"))
    return result


def known_history_for_date(customer_history: pd.DataFrame, current_date) -> pd.DataFrame:
    prior = customer_history[customer_history["baseline_create_date"] < current_date].copy()
    if prior.empty:
        return prior

    paid_before_current = prior["clear_date"].notna() & (prior["clear_date"] < current_date)
    unpaid_as_of_current = prior["clear_date"].isna() | (prior["clear_date"] >= current_date)
    unpaid_late_days = business_days_between(prior["due_in_date"], current_date)
    # [기존 코드 - 주석처리] known_late_unpaid = unpaid_as_of_current & (unpaid_late_days >= LATE_BUSINESS_DAYS)
    # [FIXED] 고침: target과 동일하게 > 기준을 적용해 정확히 5영업일 지연 건을 정상으로 판정한다.
    known_late_unpaid = unpaid_as_of_current & (unpaid_late_days > LATE_BUSINESS_DAYS)

    # 이미 결제됐거나 현재 시점에 지연 확정인 과거 인보이스만 사용한다.
    known = prior[paid_before_current | known_late_unpaid].copy()
    if known.empty:
        return known

    paid_late_days = business_days_between(known["due_in_date"], known["clear_date"])
    known_late_unpaid = known_late_unpaid.reindex(known.index).fillna(False)

    known["_known_late_target"] = np.where(
        known_late_unpaid,
        1.0,
        known["target"].astype(float),
    )
    known["_known_late"] = np.where(
        known_late_unpaid,
        1.0,
        # [기존 코드 - 주석처리] (paid_late_days >= LATE_BUSINESS_DAYS).astype(float),
        # [FIXED] 고침: 결제 완료된 과거 건도 target과 동일하게 > 기준으로 연체 여부를 계산한다.
        (paid_late_days > LATE_BUSINESS_DAYS).astype(float),
    )
    known["_days_late_as_of_current"] = np.where(
        known_late_unpaid,
        unpaid_late_days.reindex(known.index),
        paid_late_days,
    )

    # 최근 이력 정렬에 사용할 '지연 여부를 알게 된 날짜'를 만든다.
    known["_known_event_date"] = known["clear_date"]
    known.loc[known_late_unpaid, "_known_event_date"] = add_business_days(
        known.loc[known_late_unpaid, "due_in_date"],
        # [기존 코드 - 주석처리] LATE_BUSINESS_DAYS,
        # [FIXED] 고침: > 5 기준에서는 6영업일째부터 연체 확정이므로 +1을 더한다.
        LATE_BUSINESS_DAYS + 1,
    )
    return known


def weighted_late_rate(known: pd.DataFrame, current_date) -> float:
    event_month = known["_known_event_date"].dt.to_period("M")
    current_month = pd.Period(current_date, freq="M")
    month_gap = np.array([current_month.ordinal - month.ordinal for month in event_month], dtype=float)
    weights = TIME_DECAY_LAMBDA ** month_gap
    return float(np.average(known["_known_late"].astype(float), weights=weights))


def add_customer_history_features(current_df: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
    features = [
        "cust_allowed_pay_days_late_rate_past",
        "ratio_paid_invoices_late_past",
        "avg_days_late_paid_late_past",
        "sum_outstanding_amount_past",
        "recent_5_late_rate",
        "recent_10_late_rate",
        "recent_20_late_rate",
        "late_rate_time_decay_lambda_0_8",
    ]
    current_df[features] = 0.0

    for cust, current_rows_for_customer in current_df.groupby("cust_number", sort=False):
        customer_history = history_df[history_df["cust_number"].eq(cust)]

        for current_date, current_rows in current_rows_for_customer.groupby("baseline_create_date", sort=True):
            known = known_history_for_date(customer_history, current_date)
            if not known.empty:
                rate_by_allowed_days = known.groupby("Allowed_Pay_Days")["_known_late_target"].mean()
                current_df.loc[current_rows.index, "cust_allowed_pay_days_late_rate_past"] = (
                    current_rows["Allowed_Pay_Days"].map(rate_by_allowed_days).fillna(0.0).astype(float)
                )
                current_df.loc[current_rows.index, "ratio_paid_invoices_late_past"] = float(known["_known_late"].mean())

                known_late = known[known["_known_late"].eq(1.0)]
                if not known_late.empty:
                    current_df.loc[current_rows.index, "avg_days_late_paid_late_past"] = float(
                        known_late["_days_late_as_of_current"].mean()
                    )

                known_sorted = known.sort_values(["_known_event_date", "baseline_create_date", "due_in_date", "clear_date"])
                for window in RECENT_HISTORY_WINDOWS:
                    current_df.loc[current_rows.index, f"recent_{window}_late_rate"] = float(
                        known_sorted.tail(window)["_known_late"].mean()
                    )
                current_df.loc[current_rows.index, "late_rate_time_decay_lambda_0_8"] = weighted_late_rate(known_sorted, current_date)

            # 현재 시점에 아직 결제되지 않은 과거 인보이스 금액 합계.
            outstanding = customer_history[
                (customer_history["baseline_create_date"] < current_date)
                & (customer_history["clear_date"].isna() | (customer_history["clear_date"] >= current_date))
            ]
            current_df.loc[current_rows.index, "sum_outstanding_amount_past"] = max(
                float(outstanding["amount_in_usd"].sum()),
                0.0,
            )

    return current_df


def first_transaction_late_rate(train_df: pd.DataFrame) -> float:
    train_sorted = train_df.sort_values(["cust_number", "baseline_create_date", "due_in_date", "clear_date"])
    first_rows = train_sorted.groupby("cust_number", sort=False).head(1)
    return float(first_rows["target"].mean())


def add_cleared_customer_features(current_df: pd.DataFrame, history_df: pd.DataFrame, baseline_late_rate: float) -> pd.DataFrame:
    features = [
        "current_transaction_count",
        "cleared_count",
        "is_new_customer",
        "is_last_late",
        "recent_3_late_rate",
    ]
    current_df[features] = 0.0

    for cust, current_rows_for_customer in current_df.groupby("cust_number", sort=False):
        customer_history = history_df[history_df["cust_number"].eq(cust)].copy()

        for current_date, current_rows in current_rows_for_customer.groupby("baseline_create_date", sort=True):
            prior_issued = customer_history[customer_history["baseline_create_date"] < current_date]
            current_df.loc[current_rows.index, "current_transaction_count"] = np.arange(
                len(prior_issued) + 1,
                len(prior_issued) + len(current_rows) + 1,
            )

            prior_cleared = customer_history[
                customer_history["clear_date"].notna() & (customer_history["clear_date"] < current_date)
            ].sort_values(["clear_date", "baseline_create_date", "due_in_date"])

            cleared_count = len(prior_cleared)
            current_df.loc[current_rows.index, "cleared_count"] = cleared_count
            current_df.loc[current_rows.index, "is_new_customer"] = int(cleared_count < 3)

            if prior_cleared.empty:
                current_df.loc[current_rows.index, "is_last_late"] = baseline_late_rate
                current_df.loc[current_rows.index, "recent_3_late_rate"] = baseline_late_rate
            else:
                current_df.loc[current_rows.index, "is_last_late"] = float(prior_cleared["target"].iloc[-1])
                current_df.loc[current_rows.index, "recent_3_late_rate"] = float(prior_cleared["target"].tail(3).mean())

    current_df["current_transaction_count"] = current_df["current_transaction_count"].astype(int)
    current_df["cleared_count"] = current_df["cleared_count"].astype(int)
    current_df["is_new_customer"] = current_df["is_new_customer"].astype(int)
    return current_df


train_eda_revised = add_customer_history_features(train_eda_revised, train_eda_revised)
# [기존 코드 - 주석처리] test_history = pd.concat([train_eda_revised, test_eda_revised], axis=0, ignore_index=True)
# [기존 코드 - 주석처리] test_eda_revised = add_customer_history_features(test_eda_revised, test_history)
# [FIXED] 고침: holdout 평가에서 test clear_date/target이 다른 test 행의 피처에 섞이지 않도록 train 이력만 사용한다.
test_history = train_eda_revised.copy()
test_eda_revised = add_customer_history_features(test_eda_revised, test_history)

new_customer_baseline = first_transaction_late_rate(train_eda_revised)
train_eda_revised = add_cleared_customer_features(train_eda_revised, train_eda_revised, new_customer_baseline)
# [기존 코드 - 주석처리] test_history = pd.concat([train_eda_revised, test_eda_revised], axis=0, ignore_index=True)
# [기존 코드 - 주석처리] test_eda_revised = add_cleared_customer_features(test_eda_revised, test_history, new_customer_baseline)
# [FIXED] 고침: cleared 기반 test 피처도 test 라벨 정보 없이 train 이력만으로 계산한다.
test_history = train_eda_revised.copy()
test_eda_revised = add_cleared_customer_features(test_eda_revised, test_history, new_customer_baseline)

history_features = [
    "cust_allowed_pay_days_late_rate_past",
    "ratio_paid_invoices_late_past",
    "avg_days_late_paid_late_past",
    "sum_outstanding_amount_past",
    "recent_3_late_rate",
    "recent_5_late_rate",
    "recent_10_late_rate",
    "recent_20_late_rate",
    "late_rate_time_decay_lambda_0_8",
    "current_transaction_count",
    "cleared_count",
    "is_new_customer",
    "is_last_late",
]
print(f"신규 고객 기준 연체율: {new_customer_baseline:.4f}")
train_eda_revised[history_features].describe()

신규 고객 기준 연체율: 0.1497


,cust_allowed_pay_days_late_rate_past,ratio_paid_invoices_late_past,avg_days_late_paid_late_past,sum_outstanding_amount_past,recent_3_late_rate,recent_5_late_rate,recent_10_late_rate,recent_20_late_rate,late_rate_time_decay_lambda_0_8,current_transaction_count,cleared_count,is_new_customer,is_last_late
count,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000
mean,0.058288,0.059152,15.072908,725.144642,0.051533,0.072054,0.064529,0.060965,0.060787,983.396029,898.730767,0.116932,0.047788
std,0.175461,0.174964,16.257310,1048.036935,0.166076,0.200997,0.184410,0.176347,0.176774,1819.173921,1720.458276,0.321344,0.188602
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,34.669616,0.000000,0.000000,0.000000,0.000000,0.000000,27.000000,18.000000,0.000000,0.000000
50%,0.007274,0.007410,10.000000,160.514769,0.000000,0.000000,0.000000,0.000000,0.009073,134.000000,107.000000,0.000000,0.000000
75%,0.023077,0.026455,23.666667,639.039811,0.000000,0.000000,0.000000,0.050000,0.027174,670.000000,571.000000,0.000000,0.000000
max,1.000000,1.000000,86.000000,3597.627149,1.000000,1.000000,1.000000,1.000000,1.000000,7347.000000,6978.000000,1.000000,1.000000


## 거시경제 변수 통합

In [ ]:
def add_market_macro_features(train_df: pd.DataFrame, test_df: pd.DataFrame, macro_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    macro = pd.read_csv(macro_path)
    market_cols = [col for col in macro.columns if col != "year_month"]
    drop_cols = ["year_month", *market_cols]

    result = []
    for df in [train_df, test_df]:
        featured = df.drop(columns=[col for col in drop_cols if col in df.columns]).copy()
        featured["year_month"] = featured["baseline_create_date"].dt.to_period("M").astype(str)
        featured = featured.merge(macro, on="year_month", how="left")
        result.append(featured)
    return result[0], result[1]


def add_quarterly_macro_features(train_df: pd.DataFrame, test_df: pd.DataFrame, macro_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    macro = pd.read_csv(macro_path)
    macro["year_quarter"] = macro["year_quarter"].astype(str)
    quarter_cols = [col for col in macro.columns if col not in ["quarter_end_date"]]

    result = []
    for df in [train_df, test_df]:
        featured = df.drop(columns=[col for col in quarter_cols if col in df.columns]).copy()
        featured["year_quarter"] = featured["baseline_create_date"].dt.to_period("Q").astype(str)
        featured = featured.merge(macro[quarter_cols], on="year_quarter", how="left")
        result.append(featured)
    return result[0], result[1]


train_eda_revised, test_eda_revised = add_market_macro_features(train_eda_revised, test_eda_revised, MACRO_MARKET_PATH)
train_eda_revised, test_eda_revised = add_quarterly_macro_features(train_eda_revised, test_eda_revised, MACRO_QUARTER_PATH)

macro_features = [
    "vix", "hy_spread_proxy", "vix_lag1", "vix_lag2", "vix_lag3",
    "hy_spread_proxy_lag1", "hy_spread_proxy_lag2", "hy_spread_proxy_lag3",
    "gdp_growth", "unemployment", "cpi_yoy", "fed_rate", "wti_oil", "dxy", "retail_sales",
]
train_eda_revised[macro_features].describe()

,vix,hy_spread_proxy,vix_lag1,vix_lag2,vix_lag3,hy_spread_proxy_lag1,hy_spread_proxy_lag2,hy_spread_proxy_lag3,gdp_growth,unemployment,cpi_yoy,fed_rate,wti_oil,dxy,retail_sales
count,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,31326.000000,3.132600e+04
mean,15.632531,59.606932,16.541312,16.991734,17.271293,59.079669,58.597623,58.187915,3.449227,3.683017,1.785438,2.224970,57.044241,115.638294,1.512545e+06
std,2.181664,1.368993,3.223789,3.312393,3.481662,1.685541,1.732515,1.669907,0.926837,0.113848,0.136318,0.262403,1.962743,0.787809,2.200975e+04
min,12.523500,55.424639,12.949048,12.949048,12.910526,55.424639,55.424639,55.424639,0.600000,3.600000,1.630023,1.643333,54.823115,114.554878,1.481483e+06
25%,14.485238,58.579473,14.485238,14.485238,14.485238,57.993176,56.699214,56.699214,2.500000,3.600000,1.630023,2.190000,54.823115,114.554878,1.481483e+06
50%,15.466522,59.869106,15.559000,15.836000,16.721818,59.434668,59.194921,58.579473,3.400000,3.633333,1.749400,2.396667,56.336349,115.429475,1.509335e+06
75%,16.721818,60.442122,18.979091,19.389048,19.389048,60.347660,59.869106,59.434668,4.800000,3.866667,1.822384,2.403333,59.880159,116.474905,1.530934e+06
max,24.953158,61.364294,24.953158,24.953158,24.953158,61.163535,61.163535,60.442122,4.800000,3.866667,2.213519,2.403333,59.965574,116.474905,1.540729e+06


# 4. 저장 및 검증

In [ ]:
train_eda_revised.to_csv(TRAIN_OUT, index=False)
test_eda_revised.to_csv(TEST_OUT, index=False)

for name, path, df in [
    ("train", TRAIN_OUT, train_eda_revised),
    ("test", TEST_OUT, test_eda_revised),
]:
    print(f"Saved {name}: {path}")
    # [기존 코드 - 주석처리] print(f"rows={len(df):,}, cols={len(df.columns)}, business_day_gap={'business_day_gap' in df.columns}")
    # [FIXED] 고침: 실제 생성 컬럼명은 business_days_late이므로 올바른 컬럼을 검증한다.
    print(f"rows={len(df):,}, cols={len(df.columns)}, business_days_late={'business_days_late' in df.columns}")

Saved train: /content/train_eda_revised.csv
rows=31,326, cols=57, business_days_late=True
Saved test: /content/test_eda_revised.csv
rows=7,832, cols=57, business_days_late=True
